# App Clustering Pipeline: BERTopic Approach

This notebook clusters mobile subscription apps using **BERTopic** — a topic modelling framework that combines three stages:

1. **Dimensionality reduction** (UMAP) — compress 768-dim embeddings to 2D while preserving local neighbourhood structure
2. **Density-based clustering** (HDBSCAN) — find clusters of arbitrary shape; label outliers as noise (`topic = -1`)
3. **Topic representation** (c-TF-IDF) — extract the most distinctive keywords per cluster to produce human-readable topic labels

**Key advantages over KMeans / graph-based approaches:**
- No need to specify the number of clusters — HDBSCAN discovers it from the data
- Produces interpretable keyword labels for each topic automatically
- Handles outliers explicitly rather than forcing every app into a cluster

**Embeddings are reused** from `embeddings_features.npy` (computed in `kmeans.ipynb`) — no re-encoding needed.

In [20]:
import pandas as pd
import numpy as np
import umap
from bertopic import BERTopic
import hdbscan

## 1. Data Loading

Load the raw dataset and drop screenshot URL columns that carry no semantic signal.

In [21]:
df = pd.read_json('subscription_apps.json')
df.head()

,trackName,description,screenshotUrls,ipadScreenshotUrls,appletvScreenshotUrls,overview,features
0,Logo Maker - AI Art Design,Logo Maker AI-Powered Logo Design Made Easy\nC...,[https://is1-ssl.mzstatic.com/image/thumb/Purp...,[],[],Logo Maker is an AI-powered logo design app th...,[AI-Powered Logo Generation with customizable ...
1,Arvin® - AI Logo Maker,Arvin®: Your AI-Powered Logo Maker\n\nTransfor...,[https://is1-ssl.mzstatic.com/image/thumb/Purp...,[],[],Arvin® is an AI-powered logo maker designed fo...,"[AI-generated logo design, Customizable letter..."
2,AI Logo Generator & Design,AI LOGO MAKER: GENERATE STUNNING LOGOS IN SECO...,[https://is1-ssl.mzstatic.com/image/thumb/Purp...,[https://is1-ssl.mzstatic.com/image/thumb/Purp...,[],AI Logo Generator is a powerful app designed f...,"[AI-powered logo generation in seconds, Over 5..."
3,Logo Maker _ AI Design Creator,Logo Maker allows you to create the logo of yo...,[https://is1-ssl.mzstatic.com/image/thumb/Purp...,[https://is1-ssl.mzstatic.com/image/thumb/Purp...,[],Logo Maker is a user-friendly app designed for...,"[Access to over 7000 logo templates, Diverse c..."
4,Logo Maker Shop: AI Creator,Logo Maker Shop iOS app lets you create a stun...,[https://is1-ssl.mzstatic.com/image/thumb/Purp...,[https://is1-ssl.mzstatic.com/image/thumb/Purp...,[],Logo Maker Shop is an intuitive iOS app design...,[10K+ customizable logo templates across 13 ca...


In [22]:
df.drop(columns=['screenshotUrls', 'ipadScreenshotUrls', 'appletvScreenshotUrls'], inplace=True)

## 2. Load Pre-Computed Embeddings

Embeddings were generated in `kmeans.ipynb` using `BAAI/bge-base-en` on the `features` column and saved to disk.  
Loading them here avoids re-running the expensive encoding step (~5 min).

In [23]:
# Reuse embeddings computed in kmeans.ipynb — avoids re-running the ~5 min encoding step
embeddings = np.load("embeddings_features.npy")

In [24]:
df["embedding_features"] = embeddings.tolist()
df.head()

,trackName,description,overview,features,embedding_features
0,Logo Maker - AI Art Design,Logo Maker AI-Powered Logo Design Made Easy\nC...,Logo Maker is an AI-powered logo design app th...,[AI-Powered Logo Generation with customizable ...,"[0.026286382228136063, 0.014646739698946476, -..."
1,Arvin® - AI Logo Maker,Arvin®: Your AI-Powered Logo Maker\n\nTransfor...,Arvin® is an AI-powered logo maker designed fo...,"[AI-generated logo design, Customizable letter...","[0.029506200924515724, 0.018569650128483772, 0..."
2,AI Logo Generator & Design,AI LOGO MAKER: GENERATE STUNNING LOGOS IN SECO...,AI Logo Generator is a powerful app designed f...,"[AI-powered logo generation in seconds, Over 5...","[0.02679363079369068, 0.02018563449382782, -0...."
3,Logo Maker _ AI Design Creator,Logo Maker allows you to create the logo of yo...,Logo Maker is a user-friendly app designed for...,"[Access to over 7000 logo templates, Diverse c...","[0.01727970503270626, 0.016116071492433548, 0...."
4,Logo Maker Shop: AI Creator,Logo Maker Shop iOS app lets you create a stun...,Logo Maker Shop is an intuitive iOS app design...,[10K+ customizable logo templates across 13 ca...,"[0.013803043402731419, 0.02349875494837761, 0...."


## 3. Text Preparation

BERTopic needs a text document per app to extract topic keywords via c-TF-IDF.  
The `final_text` column formats the `features` list as bullet points — the same structured representation used during embedding.

In [25]:
def build_final_text(row):
    """Format the features list as bullet points for c-TF-IDF keyword extraction."""
    return "\n".join([f"- {f}" for f in row["features"]])

In [26]:
df["final_text"] = df.apply(build_final_text, axis=1)
df.head()

,trackName,description,overview,features,embedding_features,final_text
0,Logo Maker - AI Art Design,Logo Maker AI-Powered Logo Design Made Easy\nC...,Logo Maker is an AI-powered logo design app th...,[AI-Powered Logo Generation with customizable ...,"[0.026286382228136063, 0.014646739698946476, -...",- AI-Powered Logo Generation with customizable...
1,Arvin® - AI Logo Maker,Arvin®: Your AI-Powered Logo Maker\n\nTransfor...,Arvin® is an AI-powered logo maker designed fo...,"[AI-generated logo design, Customizable letter...","[0.029506200924515724, 0.018569650128483772, 0...",- AI-generated logo design\n- Customizable let...
2,AI Logo Generator & Design,AI LOGO MAKER: GENERATE STUNNING LOGOS IN SECO...,AI Logo Generator is a powerful app designed f...,"[AI-powered logo generation in seconds, Over 5...","[0.02679363079369068, 0.02018563449382782, -0....",- AI-powered logo generation in seconds\n- Ove...
3,Logo Maker _ AI Design Creator,Logo Maker allows you to create the logo of yo...,Logo Maker is a user-friendly app designed for...,"[Access to over 7000 logo templates, Diverse c...","[0.01727970503270626, 0.016116071492433548, 0....",- Access to over 7000 logo templates\n- Divers...
4,Logo Maker Shop: AI Creator,Logo Maker Shop iOS app lets you create a stun...,Logo Maker Shop is an intuitive iOS app design...,[10K+ customizable logo templates across 13 ca...,"[0.013803043402731419, 0.02349875494837761, 0....",- 10K+ customizable logo templates across 13 c...


## 4. BERTopic Model Configuration

BERTopic is composed of three pluggable sub-models. Each is configured separately and passed in:

### UMAP — Dimensionality Reduction
Reduces 768-dim embeddings to 2D before clustering.  
- `n_neighbors=5` — small value focuses on tight local structure, helping HDBSCAN find fine-grained clusters
- `min_dist=0.0` — allows points to pack tightly together in 2D, improving HDBSCAN cluster density detection
- `metric='cosine'` — consistent with how the embeddings were trained

### HDBSCAN — Density-Based Clustering
Finds clusters of arbitrary shape without requiring a pre-specified `k`.
- `min_cluster_size=2` — allows clusters as small as 2 apps (aggressive; catches niche competitors)
- `min_samples=2` — controls noise sensitivity; lower = fewer noise points labelled `-1`
- `metric='euclidean'` — applied to the already-reduced 2D UMAP output
- `cluster_selection_method='eom'` — Excess of Mass: better at finding clusters of varying density
- `prediction_data=True` — enables assigning new points to existing topics later

### BERTopic
Orchestrates the full pipeline: UMAP → HDBSCAN → c-TF-IDF representation.
- `embedding_model=None` — we pass pre-computed embeddings directly to `fit_transform`, skipping internal encoding
- `calculate_probabilities=True` — returns per-document topic probability scores alongside hard assignments

In [27]:
# --- UMAP: compress 768-dim embeddings → 2D for HDBSCAN ---
# n_neighbors=5: tight local focus → fine-grained cluster boundaries
# min_dist=0.0: points can overlap in 2D → denser, more separable clusters for HDBSCAN
umap_model = umap.UMAP(
    n_neighbors=5,
    n_components=2,
    min_dist=0.0,
    metric='cosine',
    random_state=42,
)

# --- HDBSCAN: density-based clustering on the 2D UMAP output ---
# min_cluster_size=2: accept clusters as small as 2 apps (catches rare niches)
# min_samples=2: low noise sensitivity → fewer apps labelled as outliers (-1)
# cluster_selection_method='eom': Excess of Mass handles clusters of varying density
hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=2,
    min_samples=2,
    metric='euclidean',          # applied to 2D UMAP space, not raw embeddings
    cluster_selection_method='eom',
    prediction_data=True,        # enables future inference on new documents
)

# --- BERTopic: orchestrates UMAP → HDBSCAN → c-TF-IDF ---
# embedding_model=None: we supply pre-computed embeddings directly to fit_transform
# calculate_probabilities=True: get soft topic assignments alongside hard labels
topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    embedding_model=None,
    calculate_probabilities=True,
    verbose=True,
)

In [28]:
# Pass pre-computed embeddings directly — BERTopic skips internal encoding
# topics: integer cluster label per app (-1 = HDBSCAN outlier / noise)
# probs:  probability of belonging to the assigned topic
topics, probs = topic_model.fit_transform(df["final_text"].tolist(), embeddings=embeddings)

df['cluster'] = topics

n_topics = len(set(topics)) - (1 if -1 in topics else 0)
n_noise  = topics.count(-1)
print(f"Topics found: {n_topics}  |  Noise (topic=-1): {n_noise} ({n_noise/len(topics)*100:.1f}%)")

2026-03-28 18:47:29,115 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-28 18:47:36,119 - BERTopic - Dimensionality - Completed ✓
2026-03-28 18:47:36,124 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-28 18:49:11,068 - BERTopic - Cluster - Completed ✓
2026-03-28 18:49:11,131 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-28 18:49:12,742 - BERTopic - Representation - Completed ✓


Topics found: 580  |  Noise (topic=-1): 695 (16.3%)


## 5. Fit BERTopic & Assign Topics

Run the full pipeline: UMAP → HDBSCAN → c-TF-IDF.  
`topics` contains one integer label per app; `probs` contains the probability of each app belonging to its assigned topic.  
Apps labelled `−1` are HDBSCAN outliers — too dissimilar to any cluster to be assigned.

In [29]:
# Summary table: one row per topic, sorted by size (largest first)
# Topic -1 is always first — it represents HDBSCAN outliers, not a real cluster
topic_info = topic_model.get_topic_info()
print(topic_info.head(10))

   Topic  Count                                        Name  \
0     -1    695         -1_content_workout_personalized_for   
1      0     57              0_weather_radar_forecasts_wind   
2      1     36     1_collages_layouts_slideshows_slideshow   
3      2     34      2_duplicate_duplicates_cleaner_cleanup   
4      3     33                3_qr_codes_barcodes_scanning   
5      4     28                  4_colorize_restore_old_fix   
6      5     25         5_duplicate_delete_similar_contacts   
7      6     25  6_personality_individuals_matching_profile   
8      7     24                  7_spam_call_unknown_caller   
9      8     23           8_accessories_fashion_rom_outfits   

                                      Representation  \
0  [content, workout, personalized, for, progress...   
1  [weather, radar, forecasts, wind, severe, prec...   
2  [collages, layouts, slideshows, slideshow, add...   
3  [duplicate, duplicates, cleaner, cleanup, comp...   
4  [qr, codes, barcodes, s